# Hyperparameter Optimization & Tuning
**Level:** Advanced | **Interactive Machine Learning Curriculum**

> Grid Search, Random Search, Bayesian Optimization, Optuna (TPE), Successive Halving, and Hyperband.

---

### 📖 Conceptual Foundations

#### Parameters vs Hyperparameters

- **Model Parameters ($\theta$):** Learned automatically from training data via optimization algorithms (e.g., weights $W$ and biases $b$).
- **Hyperparameters ($\lambda$):** External structural configurations set *prior* to training that govern the learning process (e.g., learning rate $\alpha$, batch size, weight decay $\lambda$, network depth, dropout probability $p$).

HPO seeks $\lambda^* = \arg\min_{\lambda \in \Lambda} \mathcal{L}_{val}(M(\lambda))$. Because evaluating $\mathcal{L}_{val}(\lambda)$ requires a full model training run, evaluations are **black-box, non-differentiable, and computationally expensive**.


#### Comparison of Hyperparameter Search Strategies

| Algorithm | Search Paradigm | Exploration vs Exploitation | Curse of Dimensionality | Efficiency Rating |
| --- | --- | --- | --- | --- |
| Grid Search | Exhaustive evaluation of an evenly spaced cartesian grid of points. | Zero exploitation; purely fixed exploration. | Exponentially crippled ($O(k^d)$ trials for $d$ hyperparameters with $k$ values). | Extremely Low for deep learning. |
| Random Search (Bergstra & Bengio, 2012) | Independent random uniform/log-uniform sampling from defined probability distributions. | Pure exploration without exploitation. | Immune to low effective dimensionality; samples unique values for every parameter. | Moderate; 10x-100x more efficient than Grid Search. |
| Bayesian Optimization (Gaussian Process) | Fits a probabilistic surrogate model (GP) to past $(x, y)$ pairs and optimizes an Acquisition Function (Expected Improvement, UCB). | Optimal theoretical balance of exploration and exploitation. | Degrades in high dimensions ($d > 20$); GP fitting scales cubically $O(N^3)$. | Very High for expensive continuous functions. |
| Tree-structured Parzen Estimator (TPE / Optuna) | Models $P(x|y < y^*)$ and $P(x|y \ge y^*)$ using kernel density estimators, maximizing $\frac{\ell(x)}{g(x)}$. | Strong adaptive exploration/exploitation balance. | Handles conditional, discrete, categorical, and high-dimensional spaces robustly. | State-of-the-art industry standard (Optuna). |
| Hyperband / ASHA | Bandit-based multi-fidelity optimization that trains many configurations for few epochs and aggressively prunes underperformers (Successive Halving). | Allocates budget strictly to promising candidates. | Highly scalable to thousands of parallel workers. | Highest compute efficiency for deep networks. |


#### Why Random Search Crushes Grid Search (Bergstra & Bengio, 2012)

In high-dimensional hyperparameter spaces, only a small subset of hyperparameters significantly impact performance (e.g., learning rate matters 100x more than momentum). In a $3 \times 3$ grid of 9 trials, grid search only tests **3 distinct values** of the critical parameter. In contrast, 9 random search trials test **9 distinct values** across the entire distribution!


#### Successive Halving & Hyperband Pruning

Evaluating bad hyperparameter sets to completion wastes 90% of GPU resources:
1. **Successive Halving:** Start $N$ configurations with an initial budget $R$ (e.g. 1 epoch). Evaluate validation loss, keep the top $1 / \eta$ (e.g. top $1/3$), discard the rest, and allocate $\eta \times R$ budget. Repeat until 1 winner remains.
2. **Hyperband:** Balances early aggressive exploration against conservative late exploitation by running Successive Halving brackets with different initial configurations $N$ and budgets $R$.



In [ ]:
# %matplotlib inline
import numpy as np

# Simulated hyperparameter evaluation function
def mock_train_model(lr, weight_decay, dropout):
    # Synthetic validation loss with noise
    optimal_lr = 3e-4
    optimal_wd = 1e-4
    loss = (np.log10(lr) - np.log10(optimal_lr))**2 + \
           (np.log10(weight_decay) - np.log10(optimal_wd))**2 + \
           (dropout - 0.2)**2 + 0.15
    return float(loss + np.random.normal(0, 0.01))

# Random search loop demonstrating log-uniform sampling
np.random.seed(42)
best_loss = float('inf')
best_config = {}

print('Running 10 trials of Random Search with Log-Uniform Priors:')
for trial in range(1, 11):
    # Sample log-uniformly: 10^U(min, max)
    lr = 10 ** np.random.uniform(-5, -2) # [1e-5, 1e-2]
    weight_decay = 10 ** np.random.uniform(-6, -2) # [1e-6, 1e-2]
    dropout = np.random.uniform(0.0, 0.5)
    
    val_loss = mock_train_model(lr, weight_decay, dropout)
    if val_loss < best_loss:
        best_loss = val_loss
        best_config = {'lr': lr, 'wd': weight_decay, 'dropout': dropout}
    print(f'Trial {trial:02d}: lr={lr:.2e}, wd={weight_decay:.2e}, drop={dropout:.2f} -> Val Loss: {val_loss:.4f}')

print(f'\nBest Val Loss: {best_loss:.4f}')
print(f'Best Config: {best_config}')


---

### 🎯 Hands-On Experiments & Challenges
1. **Experiment with parameters:** Modify the hyperparameters or dataset sizes in the cell above.
2. **Add visual diagnostics:** Plot loss curves or feature distributions using Matplotlib/Seaborn.
3. **Test with new data:** Try running the model on synthetic noisy data or custom inputs.


In [ ]:
# Your custom experiment code here:

